# اليوم الثاني — Lab مستمر من البيانات الخام إلى أول نموذج
افتح هذا الدفتر في بداية اليوم واتركه مفتوحًا. ستتوقف عند نهاية كل محطة، تعود إلى الشرح، ثم تكمل من المكان نفسه. لا تشغّل الدفتر كاملًا دفعة واحدة.

## البداية — ارفع ملف منافذ
اختر الملف `manafeth_customers.csv`. سنحتفظ بـ`df` كما هو، ونستخدم `df_clean` لأي تعديل.

In [ ]:
import io
import pandas as pd
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
df_clean = df.copy()
print('تم تحميل الملف:', file_name)
print('حجم الجدول:', df.shape)

# المحطة 1 — افهم الجدول قبل تنظيفه
شغّل الخلايا، ثم اكتب: ماذا يمثل الصف؟ وما معنى `churned_30d`؟

In [ ]:
display(df.head())
print('عدد الصفوف والأعمدة:', df.shape)
print('
أنواع الأعمدة:')
print(df.dtypes)
print('
أسماء الأعمدة:')
print(df.columns.tolist())

In [ ]:
row_meaning = ''
target_meaning = ''
print('الصف يمثل:', row_meaning)
print('الهدف يعني:', target_meaning)

## نقطة تحقق 1
لا تكمل قبل أن تستطيع تفسير الصف والهدف وثلاثة أعمدة على الأقل. عد الآن إلى شرائح الجودة والمعالجة المسبقة.

# المحطة 2 — افحص الجودة واتخذ قرارات التنظيف
نحوّل التواريخ، نوحّد النصوص، ثم نفحص الفقد والتكرار والنطاق والقيم المرتفعة. لا نحذف قيمة فقط لأنها كبيرة.

In [ ]:
date_cols = [c for c in ['signup_date', 'snapshot_date'] if c in df_clean.columns]
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

if 'city' in df_clean.columns:
    df_clean['city'] = df_clean['city'].astype('string').str.strip().str.title()

print(df_clean[date_cols].dtypes if date_cols else 'لا توجد أعمدة تاريخ')
print(df_clean['city'].value_counts(dropna=False).head(10) if 'city' in df_clean else '')

In [ ]:
missing = df_clean.isna().mean().mul(100).round(1).sort_values(ascending=False)
display(missing[missing > 0].rename('نسبة_الفقد_%').to_frame())
print('عدد الصفوف المكررة بالكامل:', df_clean.duplicated().sum())

if 'avg_rating' in df_clean.columns:
    invalid_rating = (~df_clean['avg_rating'].between(1, 5)) & df_clean['avg_rating'].notna()
    print('تقييمات خارج 1–5:', invalid_rating.sum())

if 'avg_basket_sar' in df_clean.columns:
    display(df_clean.nlargest(5, 'avg_basket_sar')[['customer_id', 'avg_basket_sar']])

In [ ]:
cleaning_decisions = pd.DataFrame({
    'الحالة': ['avg_rating مفقود', 'last_promo_used مفقود', 'سلة مرتفعة', 'صف مكرر'],
    'القرار': ['', '', '', ''],
    'السبب': ['', '', '', ''],
)
cleaning_decisions

## نقطة تحقق 2
اعرض جدول قرارات التنظيف على زميلك. إذا كان السبب غير واضح، لا تنفذ الحذف أو التعويض بعد. عد الآن إلى شرائح الخصائص والهدف والتسرب.

# المحطة 3 — جهّز X وy وامنع التسرب
`y` هي الإجابة. `X` يحتوي فقط المعلومات المتاحة وقت إصدار التوقع.

In [ ]:
y = df_clean['churned_30d'].copy()
drop_cols = [
    'customer_id',
    'churned_30d',
    'refund_issued',
    'support_ticket_after_snapshot',
    'next_month_orders',
    'signup_date',
    'snapshot_date',
]
X = df_clean.drop(columns=drop_cols, errors='ignore')
print('X:', X.shape)
print('y:', y.shape)
print('نسبة الفئة 1:', round(y.mean(), 3))

In [ ]:
forbidden = {'customer_id','churned_30d','refund_issued','support_ticket_after_snapshot','next_month_orders'}
leaked = forbidden.intersection(X.columns)
assert not leaked, f'توجد أعمدة ممنوعة داخل X: {leaked}'
print('نجح اختبار التسرب ✅')

## نقطة تحقق 3
اشرح لشخص آخر لماذا لا يظهر `churned_30d` داخل X رغم أننا سنحتاجه أثناء التدريب. عد الآن إلى شرائح التقسيم وPreprocessing وPipeline.

# المحطة 4 — قسّم البيانات وابنِ Pipeline
نتعلم جميع قواعد المعالجة من التدريب فقط، ثم نطبقها على الاختبار دون `fit` جديد.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)
print('التدريب:', X_train.shape, 'نسبة 1 =', round(y_train.mean(), 3))
print('الاختبار:', X_test.shape, 'نسبة 1 =', round(y_test.mean(), 3))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore')),
])
prep = ColumnTransformer([
    ('num', num_pipe, numeric),
    ('cat', cat_pipe, categorical),
])
print('الأعمدة الرقمية:', numeric)
print('الأعمدة الفئوية:', categorical)

## نقطة تحقق 4
أشر إلى موضع التعويض والترميز والتحجيم داخل Pipeline. عد الآن إلى شرائح Baseline والنماذج الخطية والشجرية.

# المحطة 5 — Baseline ثم نموذجان للتصنيف
نستخدم وصفة المعالجة نفسها ونغيّر النموذج فقط.

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

baseline = Pipeline([('prep', clone(prep)), ('model', DummyClassifier(strategy='most_frequent'))])
logistic = Pipeline([('prep', clone(prep)), ('model', LogisticRegression(max_iter=1000))])
tree = Pipeline([('prep', clone(prep)), ('model', DecisionTreeClassifier(max_depth=4, random_state=42))])

baseline.fit(X_train, y_train)
logistic.fit(X_train, y_train)
tree.fit(X_train, y_train)
print('تم تدريب النماذج الثلاثة ✅')

In [ ]:
from sklearn.metrics import accuracy_score

models = {'Baseline': baseline, 'Logistic Regression': logistic, 'Decision Tree': tree}
for name, fitted_model in models.items():
    prediction = fitted_model.predict(X_test)
    print(name, 'accuracy =', round(accuracy_score(y_test, prediction), 3))

probability = logistic.predict_proba(X_test)[:, 1]
display(pd.DataFrame({
    'الحقيقة': y_test.iloc[:10].values,
    'احتمال_التوقف': probability[:10].round(3),
}))

# التحدي الختامي
اكتب فقرة قصيرة تجيب عن الأسئلة التالية:

1. ما الهدف؟
2. ما الأعمدة التي استبعدتها ولماذا؟
3. كيف منعت التسرب؟
4. هل تفوق النموذجان على Baseline مبدئيًا؟
5. لماذا لا تكفي Accuracy لاختيار النموذج النهائي؟
6. ما الذي ستحتاج تعلمه في اليوم الثالث لاتخاذ قرار أفضل؟